# Actividad 3. Acondicionamiento final y aseguramiento de calidad del conjunto de datos

## Objetivo

Realizar el acondicionamiento técnico final del conjunto de datos enriquecido obtenido en la Actividad 2, garantizando que cumpla las condiciones necesarias para el entrenamiento y evaluación de los modelos predictivos de Machine Learning.

Durante esta actividad se verificará la calidad del conjunto de datos, se identificarán y tratarán los valores faltantes mediante un pipeline de preprocesamiento, se eliminarán variables no predictoras, se realizará la partición estratificada del conjunto de datos y se construirá un flujo de preparación reproducible utilizando herramientas de Scikit-learn. Finalmente, se evaluará la distribución de la variable objetivo para determinar la necesidad de aplicar técnicas de balanceo y se exportarán los conjuntos de datos preprocesados junto con el pipeline de preparación que será reutilizado durante las etapas de entrenamiento e interpretabilidad de los modelos.

---

## Relación con la metodología

Esta actividad corresponde principalmente a la fase **Data Preparation** de las metodologías **CRISP-DM** y **CRISP-ML(Q)**, incorporando además buenas prácticas de **MLOps** para la construcción de procesos reproducibles de preparación de datos.

A partir del conjunto de datos enriquecido generado en la Actividad 2, se desarrolló un pipeline de preprocesamiento que integra la imputación de valores faltantes, la codificación de variables categóricas mediante **One-Hot Encoding** y la preparación de los datos para su utilización en modelos supervisados.

Con el propósito de evitar fuga de información (*Data Leakage*), todas las transformaciones que requieren aprender información del conjunto de datos fueron ajustadas exclusivamente utilizando el conjunto de entrenamiento obtenido mediante una partición estratificada. Finalmente, se exportaron el pipeline de preprocesamiento, los conjuntos de entrenamiento y prueba preparados, y los artefactos necesarios para garantizar la reproducibilidad del proceso durante las etapas posteriores del proyecto.

# 3.1 Carga del conjunto de datos enriquecido

## Descripción

En esta sección se carga el conjunto de datos enriquecido generado durante la Actividad 2, el cual integra la información académica, demográfica y las variables derivadas mediante el proceso de ingeniería de características.

Este conjunto de datos constituye la base para las etapas de preprocesamiento, partición, balanceo y preparación de los datos que serán utilizados posteriormente en el entrenamiento y evaluación de los modelos Random Forest y XGBoost.

Asimismo, se importan las librerías necesarias para desarrollar las tareas de acondicionamiento de los datos siguiendo un flujo de trabajo reproducible y alineado con las buenas prácticas de Machine Learning.

In [1]:
# ==========================================
# Importación de librerías
# ==========================================

import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from imblearn.over_sampling import SMOTE

# Configuración general
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

RANDOM_STATE = 42

# ==========================================
# Carga del dataset enriquecido
# ==========================================

dataset_path = "../data/interim/dataset_enriched.csv"

dataset = pd.read_csv(dataset_path)

print("=" * 60)
print("Dataset cargado correctamente")
print("=" * 60)

print(f"Número de registros : {dataset.shape[0]:,}")
print(f"Número de variables : {dataset.shape[1]}")

dataset.head()

Dataset cargado correctamente
Número de registros : 32,593
Número de variables : 30


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,is_banked,academic_risk,score_mean,score_max,score_min,score_std,num_assessments,weighted_grade,score_trend,submission_delay,late_submission,late_submission_ratio,unfinished_tasks,total_clicks,active_days,max_clicks_day,median_clicks_day,avg_clicks_per_day
0,3733,DDD,2013J,M,South Region,HE Qualification,90-100%,55<=,0,60,N,Withdrawn,NaN,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,0.000,1,NaN,NaN,NaN,NaN,NaN
1,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,0.000,0,61.800,77.000,48.000,10.330,5,63.500,0.098,-2.600,0,0.000,0,2791.000,159.000,49.000,2.000,4.216
2,8462,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,0.000,1,87.667,93.000,83.000,5.033,3,87.250,-0.087,-0.333,1,0.333,0,646.000,56.000,16.000,1.000,2.153
3,8462,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,1.000,1,86.500,93.000,83.000,4.726,4,86.000,-0.038,-59.500,0,0.000,0,10.000,1.000,7.000,1.000,2.500
4,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0.000,0,82.000,85.000,78.000,3.082,5,82.400,0.014,-1.800,0,0.000,0,934.000,40.000,76.000,2.000,4.765


## Interpretación de resultados

El conjunto de datos enriquecido fue cargado correctamente, verificando el número de registros y variables obtenidos al finalizar la Actividad 2.

Este dataset representa la versión consolidada que será utilizada durante el proceso de preprocesamiento para el entrenamiento de los modelos de Machine Learning. A partir de este punto, todas las transformaciones se realizarán sobre este conjunto de datos, preservando la trazabilidad respecto a las etapas anteriores del proyecto.

La importación de las librerías y la configuración del entorno garantizan un flujo de trabajo reproducible, facilitando la reutilización del código durante las fases posteriores de entrenamiento, evaluación e interpretabilidad de los modelos.

# 3.2 Verificación de calidad del conjunto de datos

## Descripción

Antes de iniciar el proceso de preprocesamiento es necesario verificar la calidad del conjunto de datos enriquecido para garantizar que cumple con las condiciones requeridas para el entrenamiento de los modelos de Machine Learning.

En esta sección se realiza una revisión general del dataset, evaluando su estructura, tipos de datos, valores faltantes, registros duplicados y la distribución de la variable objetivo. Esta verificación permite identificar posibles inconsistencias que puedan afectar el desempeño de los modelos y definir las acciones de preparación necesarias en las siguientes etapas.

Esta actividad constituye una práctica de aseguramiento de la calidad de los datos, alineada con la fase **Data Preparation** de CRISP-DM y con los principios de calidad de datos promovidos por CRISP-ML(Q).

In [2]:
# ==========================================
# Verificación general del conjunto de datos
# ==========================================

print("=" * 70)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 70)

print(f"Número de registros : {dataset.shape[0]:,}")
print(f"Número de variables : {dataset.shape[1]}")

print("\nTipos de datos")

display(dataset.dtypes.to_frame(name="Tipo de dato"))

# ==========================================
# Valores faltantes
# ==========================================

missing = (
    dataset.isna()
           .sum()
           .to_frame(name="Valores faltantes")
)

missing["Porcentaje (%)"] = (
    missing["Valores faltantes"] / len(dataset)
) * 100

missing = (
    missing
    .sort_values(
        by="Valores faltantes",
        ascending=False
    )
)

print("\nResumen de valores faltantes")

display(
    missing[
        missing["Valores faltantes"] > 0
    ]
)

# ==========================================
# Registros duplicados
# ==========================================

duplicates = dataset.duplicated().sum()

print("\nNúmero de registros duplicados:", duplicates)

# ==========================================
# Distribución de la variable objetivo
# ==========================================

print("\nDistribución de la variable objetivo")

target_distribution = (
    dataset["academic_risk"]
    .value_counts()
    .sort_index()
    .rename_axis("academic_risk")
    .to_frame("Frecuencia")
)

target_distribution["Porcentaje (%)"] = (
    target_distribution["Frecuencia"]
    / len(dataset)
    * 100
).round(2)

display(target_distribution)

# ==========================================
# Resumen general
# ==========================================

quality_summary = pd.DataFrame({
    "Indicador": [
        "Número de registros",
        "Número de variables",
        "Variables con valores faltantes",
        "Registros duplicados"
    ],
    "Valor": [
        dataset.shape[0],
        dataset.shape[1],
        (missing["Valores faltantes"] > 0).sum(),
        duplicates
    ]
})

print("\nResumen de calidad")

display(quality_summary)

INFORMACIÓN GENERAL DEL DATASET
Número de registros : 32,593
Número de variables : 30

Tipos de datos


,Tipo de dato
id_student,int64
code_module,object
code_presentation,object
gender,object
region,object
highest_education,object
imd_band,object
age_band,object
num_of_prev_attempts,int64
studied_credits,int64



Resumen de valores faltantes


,Valores faltantes,Porcentaje (%)
score_trend,9308,28.558
score_std,9292,28.509
weighted_grade,9087,27.880
score_mean,6773,20.781
score_max,6773,20.781
score_min,6773,20.781
submission_delay,6751,20.713
is_banked,6750,20.710
total_clicks,3365,10.324
avg_clicks_per_day,3365,10.324



Número de registros duplicados: 0

Distribución de la variable objetivo


,Frecuencia,Porcentaje (%)
academic_risk,,
0,15385,47.200
1,17208,52.800



Resumen de calidad


,Indicador,Valor
0,Número de registros,32593
1,Número de variables,30
2,Variables con valores faltantes,13
3,Registros duplicados,0


## Interpretación de resultados

La verificación de calidad permitió confirmar la estructura general del conjunto de datos enriquecido antes de iniciar el proceso de preprocesamiento. Se revisaron las dimensiones del dataset, los tipos de datos asociados a cada variable, la presencia de valores faltantes, la existencia de registros duplicados y la distribución de la variable objetivo.

El análisis de valores faltantes permitirá identificar las variables que requieren un tratamiento adicional en la siguiente sección, mientras que la verificación de registros duplicados confirma si existen observaciones redundantes que puedan afectar el entrenamiento de los modelos.

Finalmente, el análisis de la distribución de la variable **academic_risk** proporciona una primera aproximación al equilibrio entre las clases. Esta información será utilizada posteriormente para justificar la aplicación de técnicas de balanceo, como SMOTE, únicamente sobre el conjunto de entrenamiento, evitando la fuga de información y preservando la validez del proceso de evaluación de los modelos.

# 3.3 Tratamiento final de valores faltantes

## Descripción

Una vez verificada la calidad del conjunto de datos, es necesario definir una estrategia de tratamiento para las variables que presentan valores faltantes.

A diferencia de aplicar un único método de imputación para todas las variables, en esta investigación se adopta un enfoque basado en el significado de cada atributo y en el origen de los valores faltantes. De esta manera, cada variable será tratada mediante una estrategia específica que preserve la información disponible y reduzca la posibilidad de introducir sesgos durante el entrenamiento de los modelos.

Este procedimiento forma parte de la fase **Data Preparation** de CRISP-DM y constituye una práctica recomendada por CRISP-ML(Q), ya que busca garantizar la calidad de los datos antes de la construcción del pipeline de Machine Learning.

In [16]:
# ==========================================
# Análisis de las variables con valores faltantes
# ==========================================

missing_summary = (
    dataset.isnull()
           .sum()
           .reset_index()
)

missing_summary.columns = [
    "Variable",
    "Valores faltantes"
]

missing_summary["Porcentaje (%)"] = (
    missing_summary["Valores faltantes"]
    / len(dataset)
    * 100
).round(2)

missing_summary = missing_summary[
    missing_summary["Valores faltantes"] > 0
].sort_values(
    by="Porcentaje (%)",
    ascending=False
)

# ==========================================
# Estrategia propuesta para cada variable
# ==========================================

imputation_strategy = pd.DataFrame({

    "Variable":[

        "score_mean",
        "score_max",
        "score_min",
        "weighted_grade",
        "score_std",
        "score_trend",
        "submission_delay",
        "total_clicks",
        "active_days",
        "max_clicks_day",
        "median_clicks_day",
        "avg_clicks_per_day",
        "is_banked"

    ],

    "Origen del valor faltante":[

        "Estudiante sin evaluaciones",
        "Estudiante sin evaluaciones",
        "Estudiante sin evaluaciones",
        "Estudiante sin evaluaciones",
        "Una sola evaluación o sin evaluaciones",
        "No existen suficientes evaluaciones",
        "Sin evaluaciones registradas",
        "Sin interacción registrada en el VLE",
        "Sin interacción registrada en el VLE",
        "Sin interacción registrada en el VLE",
        "Sin interacción registrada en el VLE",
        "Sin interacción registrada en el VLE",
        "Información faltante del dataset original"

    ],

        "Estrategia propuesta":[

        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento",
        "Imputación mediante la mediana dentro del pipeline de preprocesamiento"

    ]

})

display(missing_summary)

display(imputation_strategy)

,Variable,Valores faltantes,Porcentaje (%)
20,score_trend,9308,28.560
17,score_std,9292,28.510
19,weighted_grade,9087,27.880
15,score_max,6773,20.780
14,score_mean,6773,20.780
16,score_min,6773,20.780
12,is_banked,6750,20.710
21,submission_delay,6751,20.710
25,total_clicks,3365,10.320
26,active_days,3365,10.320


,Variable,Origen del valor faltante,Estrategia propuesta
0,score_mean,Estudiante sin evaluaciones,Imputación mediante la mediana dentro del pipe...
1,score_max,Estudiante sin evaluaciones,Imputación mediante la mediana dentro del pipe...
2,score_min,Estudiante sin evaluaciones,Imputación mediante la mediana dentro del pipe...
3,weighted_grade,Estudiante sin evaluaciones,Imputación mediante la mediana dentro del pipe...
4,score_std,Una sola evaluación o sin evaluaciones,Imputación mediante la mediana dentro del pipe...
5,score_trend,No existen suficientes evaluaciones,Imputación mediante la mediana dentro del pipe...
6,submission_delay,Sin evaluaciones registradas,Imputación mediante la mediana dentro del pipe...
7,total_clicks,Sin interacción registrada en el VLE,Imputación mediante la mediana dentro del pipe...
8,active_days,Sin interacción registrada en el VLE,Imputación mediante la mediana dentro del pipe...
9,max_clicks_day,Sin interacción registrada en el VLE,Imputación mediante la mediana dentro del pipe...


# 3.4 Selección y eliminación de variables no predictoras

## Descripción

Antes de iniciar la construcción del pipeline de preprocesamiento, es necesario identificar aquellas variables que no deben formar parte del proceso de entrenamiento de los modelos predictivos.

En esta etapa se eliminan los atributos que actúan como identificadores, aquellos que contienen información derivada directamente de la variable objetivo o variables cuya inclusión pueda provocar fuga de información (*Data Leakage*). Esta selección busca garantizar que los modelos aprendan únicamente a partir de información disponible antes de conocer el resultado académico del estudiante.

La eliminación de estas variables constituye una práctica recomendada dentro de la fase **Data Preparation** de CRISP-DM y contribuye a mejorar la calidad, validez y capacidad de generalización de los modelos desarrollados.

In [4]:
# ==========================================
# Variables candidatas para eliminación
# ==========================================

variables_to_remove = pd.DataFrame({

    "Variable":[

        "id_student",
        "final_result"

    ],

    "Motivo":[

        "Identificador único del estudiante",
        "Variable original utilizada para construir academic_risk"

    ],

    "Riesgo":[

        "No aporta capacidad predictiva",
        "Provoca fuga de información (Data Leakage)"

    ]

})

display(variables_to_remove)

,Variable,Motivo,Riesgo
0,id_student,Identificador único del estudiante,No aporta capacidad predictiva
1,final_result,Variable original utilizada para construir aca...,Provoca fuga de información (Data Leakage)


In [5]:
# ==========================================
# Eliminación de variables no predictoras
# ==========================================

dataset_model = dataset.drop(

    columns=[

        "id_student",
        "final_result"

    ]

).copy()

print("="*60)
print("Variables eliminadas correctamente")
print("="*60)

print(f"Número de variables originales : {dataset.shape[1]}")
print(f"Número de variables finales    : {dataset_model.shape[1]}")

Variables eliminadas correctamente
Número de variables originales : 30
Número de variables finales    : 28


In [6]:
dataset_model.head()

,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,is_banked,academic_risk,score_mean,score_max,score_min,score_std,num_assessments,weighted_grade,score_trend,submission_delay,late_submission,late_submission_ratio,unfinished_tasks,total_clicks,active_days,max_clicks_day,median_clicks_day,avg_clicks_per_day
0,DDD,2013J,M,South Region,HE Qualification,90-100%,55<=,0,60,N,NaN,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,0.000,1,NaN,NaN,NaN,NaN,NaN
1,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,0.000,0,61.800,77.000,48.000,10.330,5,63.500,0.098,-2.600,0,0.000,0,2791.000,159.000,49.000,2.000,4.216
2,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,0.000,1,87.667,93.000,83.000,5.033,3,87.250,-0.087,-0.333,1,0.333,0,646.000,56.000,16.000,1.000,2.153
3,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,1.000,1,86.500,93.000,83.000,4.726,4,86.000,-0.038,-59.500,0,0.000,0,10.000,1.000,7.000,1.000,2.500
4,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,0.000,0,82.000,85.000,78.000,3.082,5,82.400,0.014,-1.800,0,0.000,0,934.000,40.000,76.000,2.000,4.765


## Interpretación de resultados

Como parte del acondicionamiento final del conjunto de datos, se eliminaron aquellas variables que no deben intervenir en el entrenamiento de los modelos predictivos.

La variable **id_student** corresponde únicamente a un identificador único de cada estudiante y no aporta información útil para la predicción del riesgo académico.

Por otra parte, la variable **final_result** fue utilizada previamente para construir la variable objetivo **academic_risk**. Mantener esta variable dentro del conjunto de predictores introduciría fuga de información (*Data Leakage*), ya que el modelo tendría acceso directo al resultado académico que precisamente se pretende predecir.

Con esta depuración se garantiza que el proceso de entrenamiento se realice exclusivamente sobre variables predictoras disponibles antes de conocer el resultado final del estudiante, favoreciendo la validez y capacidad de generalización de los modelos.

# 3.5 Separación de variables predictoras y variable objetivo

## Descripción

Una vez eliminado el conjunto de variables que no deben participar en el proceso de modelado, se procede a separar la variable objetivo de las variables predictoras.

Esta separación constituye un paso fundamental dentro del flujo de trabajo de Machine Learning, ya que permite diferenciar la información utilizada como entrada del modelo (**X**) de la variable que se desea predecir (**y**). Mantener esta separación desde las primeras etapas del preprocesamiento favorece la construcción de un pipeline organizado, reproducible y libre de fuga de información.

En esta investigación, la variable objetivo corresponde a **academic_risk**, construida durante la Actividad 2 a partir de la variable **final_result**.

In [7]:
# ==========================================
# Separación de variables predictoras y objetivo
# ==========================================

X = dataset_model.drop(columns=["academic_risk"])

y = dataset_model["academic_risk"]

print("=" * 60)
print("Separación de variables completada")
print("=" * 60)

print(f"Número de variables predictoras : {X.shape[1]}")
print(f"Número de observaciones         : {X.shape[0]}")

print("\nVariable objetivo:")

display(
    y.value_counts()
      .rename_axis("academic_risk")
      .to_frame("Frecuencia")
)

Separación de variables completada
Número de variables predictoras : 27
Número de observaciones         : 32593

Variable objetivo:


,Frecuencia
academic_risk,
1,17208
0,15385


## Interpretación de resultados

Se realizó la separación entre las variables predictoras y la variable objetivo, obteniendo el conjunto **X**, conformado por los atributos que serán utilizados durante el entrenamiento de los modelos, y la variable **y**, correspondiente al riesgo académico.

Esta separación constituye un requisito previo para realizar la partición del conjunto de datos y construir posteriormente el pipeline de preprocesamiento. A partir de este punto, todas las transformaciones serán aplicadas únicamente sobre las variables predictoras, preservando la independencia de la variable objetivo durante el proceso de preparación de los datos.

# 3.6 División estratificada del conjunto de datos

## Descripción

Con el propósito de evaluar de forma objetiva el desempeño de los modelos predictivos, el conjunto de datos se divide en subconjuntos de entrenamiento y prueba mediante un muestreo estratificado.

La estratificación garantiza que ambas particiones conserven una distribución similar de la variable objetivo, evitando que una de las clases quede sobre o subrepresentada durante el entrenamiento o la evaluación.

Esta división se realiza antes de cualquier transformación que requiera ajustar parámetros a partir de los datos, como la imputación de valores faltantes o la codificación de variables categóricas. De esta manera se evita la fuga de información (*Data Leakage*), asegurando que el conjunto de prueba permanezca completamente independiente hasta la etapa de evaluación de los modelos.

In [8]:
# ==========================================
# División del conjunto de datos
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    stratify=y,

    random_state=RANDOM_STATE

)

print("=" * 60)
print("Partición realizada correctamente")
print("=" * 60)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

Partición realizada correctamente
X_train : (26074, 27)
X_test  : (6519, 27)
y_train : (26074,)
y_test  : (6519,)


In [9]:
# ==========================================
# Verificación de la estratificación
# ==========================================

train_distribution = (

    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)

)

test_distribution = (

    y_test
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)

)

distribution = pd.DataFrame({

    "Entrenamiento (%)": train_distribution,
    "Prueba (%)": test_distribution

})

display(distribution)

,Entrenamiento (%),Prueba (%)
academic_risk,,
0,47.200,47.200
1,52.800,52.800


## Interpretación de resultados

El conjunto de datos fue dividido en subconjuntos de entrenamiento y prueba utilizando un muestreo estratificado, asignando el 80 % de las observaciones al entrenamiento y el 20 % restante a la evaluación.

La verificación de la distribución de la variable objetivo confirma que ambas particiones mantienen proporciones similares entre las clases, garantizando que el proceso de entrenamiento y evaluación se realice sobre muestras representativas del problema de clasificación.

Asimismo, esta división constituye un paso fundamental para prevenir la fuga de información, ya que todas las transformaciones posteriores del pipeline serán ajustadas exclusivamente utilizando el conjunto de entrenamiento.

# 3.7 Identificación de variables numéricas y categóricas

## Descripción

Antes de construir el pipeline de preprocesamiento es necesario identificar las variables numéricas y categóricas presentes en el conjunto de entrenamiento. Esta clasificación permitirá aplicar diferentes transformaciones según la naturaleza de cada atributo.

Las variables numéricas serán sometidas a un proceso de imputación para tratar los valores faltantes, mientras que las variables categóricas serán imputadas y posteriormente codificadas mediante **One-Hot Encoding** para convertirlas en variables aptas para el entrenamiento de los modelos de Machine Learning.

La identificación de los tipos de variables se realiza únicamente sobre el conjunto de entrenamiento, manteniendo un flujo de trabajo alineado con las buenas prácticas de CRISP-ML(Q) y evitando cualquier posibilidad de fuga de información.

In [10]:
# ==========================================
# Identificación de variables
# ==========================================

numeric_features = X_train.select_dtypes(

    include=["int64", "float64"]

).columns.tolist()

categorical_features = X_train.select_dtypes(

    include=["object", "category"]

).columns.tolist()

print("=" * 60)
print("Resumen de variables")
print("=" * 60)

print(f"Variables numéricas   : {len(numeric_features)}")
print(f"Variables categóricas : {len(categorical_features)}")

print("\nVariables numéricas")

display(pd.DataFrame(numeric_features, columns=["Variable"]))

print("\nVariables categóricas")

display(pd.DataFrame(categorical_features, columns=["Variable"]))

Resumen de variables
Variables numéricas   : 19
Variables categóricas : 8

Variables numéricas


,Variable
0,num_of_prev_attempts
1,studied_credits
2,is_banked
3,score_mean
4,score_max
5,score_min
6,score_std
7,num_assessments
8,weighted_grade
9,score_trend



Variables categóricas


,Variable
0,code_module
1,code_presentation
2,gender
3,region
4,highest_education
5,imd_band
6,age_band
7,disability


## Interpretación de resultados

Se identificaron las variables numéricas y categóricas presentes en el conjunto de entrenamiento, permitiendo definir las transformaciones específicas que serán aplicadas a cada tipo de dato dentro del pipeline de preprocesamiento.

Esta separación constituye un paso fundamental para construir un proceso automatizado y reutilizable, ya que las variables numéricas y categóricas requieren estrategias de preparación diferentes antes de ser utilizadas por los modelos predictivos.

# 3.8 Construcción del pipeline de preprocesamiento

## Descripción

Una vez identificados los tipos de variables, se construye un pipeline de preprocesamiento utilizando las herramientas proporcionadas por Scikit-learn.

El pipeline aplica transformaciones específicas según el tipo de dato. Para las variables numéricas se emplea una estrategia de imputación basada en la mediana, mientras que para las variables categóricas se realiza la imputación mediante la categoría más frecuente y posteriormente se aplica la codificación **One-Hot Encoding**.

En esta investigación no se aplica escalamiento de las variables numéricas. Esta decisión se fundamenta en que los modelos Random Forest y XGBoost pertenecen a la familia de algoritmos basados en árboles de decisión, cuyo proceso de aprendizaje no depende de la magnitud o escala de las variables. Por tanto, omitir esta transformación evita modificaciones innecesarias y conserva la interpretabilidad de los atributos originales.

La construcción del pipeline favorece la reproducibilidad del proceso de preparación de los datos y facilita su reutilización durante el entrenamiento y la evaluación de los modelos.

In [11]:
# ==========================================
# Pipeline para variables numéricas
# ==========================================

numeric_pipeline = Pipeline(

    steps=[

        (

            "imputer",

            SimpleImputer(

                strategy="median"

            )

        )

    ]

)

# ==========================================
# Pipeline para variables categóricas
# ==========================================

categorical_pipeline = Pipeline(

    steps=[

        (

            "imputer",

            SimpleImputer(

                strategy="most_frequent"

            )

        ),

        (

            "encoder",

            OneHotEncoder(

                handle_unknown="ignore"

            )

        )

    ]

)

# ==========================================
# ColumnTransformer
# ==========================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "numeric",

            numeric_pipeline,

            numeric_features

        ),

        (

            "categorical",

            categorical_pipeline,

            categorical_features

        )

    ]

)

print("=" * 60)
print("Pipeline construido correctamente")
print("=" * 60)

preprocessor

Pipeline construido correctamente


,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## Interpretación de resultados

Se construyó un pipeline de preprocesamiento que integra las transformaciones requeridas para cada tipo de variable. Este enfoque permite automatizar el tratamiento de valores faltantes y la codificación de variables categóricas dentro de un único flujo de trabajo reproducible.

Además, al encapsular todas las transformaciones en un único objeto, se garantiza que el mismo proceso pueda aplicarse posteriormente durante el entrenamiento, la evaluación y el despliegue de los modelos, reduciendo el riesgo de inconsistencias entre diferentes etapas del proyecto.

# 3.9 Ajuste del preprocesador y transformación de los datos

## Descripción

Una vez construido el pipeline de preprocesamiento, este se ajusta exclusivamente utilizando el conjunto de entrenamiento. Posteriormente, las transformaciones aprendidas son aplicadas tanto al conjunto de entrenamiento como al conjunto de prueba.

Este procedimiento garantiza que las estadísticas utilizadas durante la imputación y la codificación sean obtenidas únicamente a partir de los datos de entrenamiento, preservando la independencia del conjunto de prueba y evitando la fuga de información.

In [12]:
# ==========================================
# Ajuste del preprocesador
# ==========================================

X_train_processed = preprocessor.fit_transform(

    X_train

)

X_test_processed = preprocessor.transform(

    X_test

)
print("="*60)
print("Número de variables después del preprocesamiento")
print("="*60)

print(X_train_processed.shape[1])

print("\n")
print("=" * 60)
print("Preprocesamiento completado")
print("=" * 60)

print(f"X_train procesado : {X_train_processed.shape}")
print(f"X_test procesado  : {X_test_processed.shape}")





Número de variables después del preprocesamiento
66


Preprocesamiento completado
X_train procesado : (26074, 66)
X_test procesado  : (6519, 66)


## Interpretación de resultados

El pipeline de preprocesamiento fue ajustado únicamente utilizando el conjunto de entrenamiento y posteriormente aplicado al conjunto de prueba. De esta manera, todas las transformaciones fueron aprendidas exclusivamente a partir de los datos disponibles durante el entrenamiento del modelo.

Este procedimiento garantiza un flujo de trabajo libre de fuga de información, asegurando que el conjunto de prueba permanezca completamente independiente hasta la etapa de evaluación de los modelos predictivos.

# 3.10 Obtención de las características generadas por el pipeline

## Descripción

Una vez aplicado el pipeline de preprocesamiento, se recuperan los nombres de las características resultantes de la transformación realizada por el **ColumnTransformer**.

Este procedimiento permite mantener la trazabilidad entre las variables originales y las nuevas características generadas durante la codificación de las variables categóricas mediante **One-Hot Encoding**. Además, disponer de esta información facilitará la interpretación de los modelos predictivos en las etapas posteriores del proyecto, especialmente durante la aplicación de técnicas de Inteligencia Artificial Explicable (XAI), como SHAP y LIME.

In [13]:
# ==========================================
# Recuperación de las variables generadas
# ==========================================

feature_names = preprocessor.get_feature_names_out()

feature_names_df = pd.DataFrame(

    feature_names,

    columns=["Variable"]

)

print("=" * 60)
print("Características generadas por el pipeline")
print("=" * 60)

print(f"Número total de variables: {len(feature_names)}")

display(feature_names_df.head(20))

Características generadas por el pipeline
Número total de variables: 66


,Variable
0,numeric__num_of_prev_attempts
1,numeric__studied_credits
2,numeric__is_banked
3,numeric__score_mean
4,numeric__score_max
5,numeric__score_min
6,numeric__score_std
7,numeric__num_assessments
8,numeric__weighted_grade
9,numeric__score_trend


## Interpretación de resultados

El pipeline generó el conjunto final de variables que será utilizado durante el entrenamiento de los modelos de Machine Learning. El incremento en el número de características respecto al conjunto original se debe a la aplicación de la codificación **One-Hot Encoding** sobre las variables categóricas.

La recuperación de los nombres de las variables permite conservar la trazabilidad del proceso de transformación y facilitará la interpretación de la importancia de las características durante la aplicación de SHAP y LIME en las etapas posteriores del proyecto.

# 3.11 Evaluación del balance de clases

## Descripción

Antes de aplicar técnicas de balanceo de clases, es necesario evaluar la distribución de la variable objetivo en el conjunto de entrenamiento.

Aunque existen técnicas como **SMOTE** para generar nuevas observaciones sintéticas, su aplicación únicamente resulta recomendable cuando existe un desequilibrio importante entre las clases. Por esta razón, primero se analiza la distribución del conjunto de entrenamiento y posteriormente se determina si el uso de técnicas de sobremuestreo es necesario.

In [14]:
# ==========================================
# Distribución del conjunto de entrenamiento
# ==========================================

train_balance = (

    y_train
    .value_counts()
    .sort_index()
    .rename_axis("academic_risk")
    .to_frame("Frecuencia")

)

train_balance["Porcentaje (%)"] = (

    train_balance["Frecuencia"]

    / len(y_train)

    * 100

).round(2)

print("=" * 60)
print("Distribución del conjunto de entrenamiento")
print("=" * 60)

display(train_balance)

difference = (

    train_balance["Porcentaje (%)"].max()

    - train_balance["Porcentaje (%)"].min()

)

print(f"\nDiferencia entre clases: {difference:.2f}%")

Distribución del conjunto de entrenamiento


,Frecuencia,Porcentaje (%)
academic_risk,,
0,12308,47.200
1,13766,52.800



Diferencia entre clases: 5.60%


## Interpretación de resultados

La distribución de la variable objetivo en el conjunto de entrenamiento evidencia una proporción cercana al equilibrio entre ambas clases, con una diferencia porcentual reducida.

Considerando este resultado, no se aplicó la técnica de sobremuestreo **SMOTE**, ya que su utilización está orientada principalmente a conjuntos de datos con un desequilibrio significativo entre categorías. Generar observaciones sintéticas sobre un conjunto que ya presenta una distribución balanceada podría introducir variabilidad innecesaria y alterar la representación original de los datos.

En consecuencia, el entrenamiento de los modelos Random Forest y XGBoost se realizará utilizando la distribución original del conjunto de entrenamiento.

# 3.12 Exportación del pipeline y de los conjuntos preprocesados

## Descripción

Como etapa final del proceso de acondicionamiento de los datos, se exportan los conjuntos de entrenamiento y prueba previamente transformados, así como el pipeline de preprocesamiento construido durante esta actividad.

La exportación de estos elementos permite reutilizar exactamente el mismo proceso de preparación durante las etapas de entrenamiento, evaluación e interpretabilidad de los modelos, garantizando la reproducibilidad del flujo de trabajo y la consistencia entre las diferentes actividades del proyecto.

In [15]:
# ==========================================
# Exportación del pipeline
# ==========================================

joblib.dump(

    preprocessor,

    "../data/processed/preprocessing_pipeline.joblib"

)

# ==========================================
# Exportación de los conjuntos procesados
# ==========================================

joblib.dump(

    X_train_processed,

    "../data/processed/X_train_processed.joblib"

)

joblib.dump(

    X_test_processed,

    "../data/processed/X_test_processed.joblib"

)

joblib.dump(

    y_train,

    "../data/processed/y_train.joblib"

)

joblib.dump(

    y_test,

    "../data/processed/y_test.joblib"

)

# ==========================================
# Exportación de los nombres de variables
# ==========================================

# Variables originales (antes del One-Hot Encoding)
joblib.dump(

    X_train.columns.tolist(),

    "../data/processed/original_feature_names.joblib"

)

# Variables finales (después del One-Hot Encoding)
joblib.dump(

    feature_names_df,

    "../data/processed/feature_names.joblib"

)

# ==========================================
# Resumen de archivos exportados
# ==========================================

exported_files = [

    "preprocessing_pipeline.joblib",
    "X_train_processed.joblib",
    "X_test_processed.joblib",
    "y_train.joblib",
    "y_test.joblib",
    "original_feature_names.joblib",
    "feature_names.joblib"

]

print("=" * 60)
print("Archivos exportados correctamente")
print("=" * 60)

for file in exported_files:
    print(f"✓ {file}")

Archivos exportados correctamente
✓ preprocessing_pipeline.joblib
✓ X_train_processed.joblib
✓ X_test_processed.joblib
✓ y_train.joblib
✓ y_test.joblib
✓ original_feature_names.joblib
✓ feature_names.joblib


## Interpretación de resultados

Se exportó el pipeline de preprocesamiento junto con los conjuntos de entrenamiento y prueba preparados durante esta actividad.

Estos recursos constituyen el entregable principal de la fase de acondicionamiento del conjunto de datos y serán reutilizados durante el entrenamiento y evaluación de los modelos Random Forest y XGBoost. Asimismo, la reutilización del mismo pipeline garantiza que todas las transformaciones aplicadas a los datos sean consistentes a lo largo del proyecto, fortaleciendo la reproducibilidad del flujo de trabajo y evitando inconsistencias entre las diferentes etapas del proceso de modelado.

# Conclusiones de la Actividad 3

Durante esta actividad se llevó a cabo el acondicionamiento final del conjunto de datos enriquecido, desarrollando un proceso de preparación orientado a garantizar la calidad de los datos y la reproducibilidad del flujo de trabajo previo al entrenamiento de los modelos predictivos.

Inicialmente se verificó la calidad del conjunto de datos mediante el análisis de su estructura, tipos de datos, valores faltantes, registros duplicados y distribución de la variable objetivo. Posteriormente, se eliminaron las variables que podían introducir fuga de información, se separaron las variables predictoras de la variable objetivo y se realizó una partición estratificada del conjunto de datos para preservar la distribución de las clases entre los conjuntos de entrenamiento y prueba.

Como parte del proceso de preprocesamiento se construyó un pipeline utilizando **Scikit-learn**, el cual integra la imputación de valores faltantes y la codificación de variables categóricas mediante **One-Hot Encoding**. El ajuste del pipeline se realizó exclusivamente sobre el conjunto de entrenamiento, garantizando un proceso libre de fuga de información y alineado con las buenas prácticas de Machine Learning y MLOps.

Asimismo, se evaluó la distribución de la variable objetivo en el conjunto de entrenamiento, observándose una diferencia reducida entre las clases. En consecuencia, no fue necesario aplicar técnicas de balanceo como **SMOTE**, preservando la distribución original de los datos y evitando la generación innecesaria de observaciones sintéticas.

Finalmente, se exportaron el pipeline de preprocesamiento, los conjuntos de entrenamiento y prueba transformados, así como los nombres de las variables originales y de las características generadas durante el proceso de codificación. Estos artefactos constituyen el entregable principal de esta actividad y servirán como base para el entrenamiento, optimización, evaluación e interpretación de los modelos **Random Forest** y **XGBoost** que serán desarrollados en las siguientes etapas del proyecto.